# This Notebook is a simple Demonstration of how to use Claude API to build a simple tool call

In [51]:
from dotenv import load_dotenv
load_dotenv()
from anthropic import Anthropic
from datetime import datetime
import json

In [3]:
client = Anthropic()
model = "claude-sonnet-4-0"

# Building Helper Functions

In [5]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text

# Execution

In [6]:
messages = []
add_user_message(messages, "Define quantum computing in one sentence")
answer = chat(messages)
print(answer)
add_assistant_message(messages, answer)
add_user_message(messages, "Write a haiku about it")
answer = chat(messages)
print(answer)

Quantum computing is a type of computation that harnesses quantum mechanical phenomena like superposition and entanglement to process information in ways that can potentially solve certain problems exponentially faster than classical computers.
Qubits dance in space,
Superposed between all states—
Computing the impossible.


# Now Lets Build a Demo Chat Module

In [15]:
messages = []
flag = True
while True:
    user_input = input("> ")
    if user_input == "exit":
        break
    print("> ", user_input)
    add_user_message(messages, user_input)
    answer = chat(messages)
    add_assistant_message(messages, answer)
    print('---')
    print(answer)
    print('---')

>  hi
---
Hello! How are you doing today? Is there anything I can help you with?
---
>  whats 2  2
---
2 + 2 = 4

If you meant a different operation, let me know! For example:
- 2 × 2 = 4
- 2 - 2 = 0
- 2 ÷ 2 = 1
---
>  I ment 2+2 sry
---
No problem at all! Yes, 2 + 2 = 4. 

Is there anything else I can help you with?
---
>  now what is its cube
---
The cube of 4 is 4³ = 4 × 4 × 4 = 64.
---


# Now Lets Ask Claude something it wasn't trained on

In [7]:
messages = []
flag = True
while True:
    user_input = input("> ")
    if user_input == "exit":
        break
    print("> ", user_input)
    add_user_message(messages, user_input)
    answer = chat(messages)
    add_assistant_message(messages, answer)
    print('---')
    print(answer)
    print('---')

>  What is the current weather in Dallas
---
I don't have access to real-time weather data, so I can't tell you the current weather conditions in Dallas. To get up-to-date weather information for Dallas, I'd recommend checking:

- Weather.com or the Weather Channel app
- National Weather Service (weather.gov)
- Local Dallas news websites
- Weather apps on your phone like AccuWeather or Weather Underground

These sources will give you current conditions, temperature, precipitation, and forecasts for Dallas, Texas.
---


# Now Lets solve these type of issues with tool use.

### To do this we must ask claude the question along with some instructions on how to pull this data.

### Lets Try an example of how to set a reminder based on user queries.

### For this lets add 3 tools. 
### 1. Gets the current time. 
### 2. Sets the Target time relative to the current.
### 3. Sets a reminder.

In [19]:
from anthropic.types import ToolParam

def get_current_time(dt_format = "%Y-%m-%d %H:%M:%S"):
    if not dt_format:
        raise ValueError("proper dt_format is required")
    return datetime.now().strftime(dt_format)


get_current_time_schema = ToolParam({
  "name": "get_current_time",
  "description": "Get the current date and time formatted according to the specified format string. Returns a formatted timestamp using Python's strftime directives.",
  "input_schema": {
    "type": "object",
    "properties": {
      "dt_format": {
        "type": "string",
        "description": "Python strftime format string for the output. Common directives: %Y (4-digit year), %m (month), %d (day), %H (24hr hour), %M (minute), %S (second). Example: '%Y-%m-%d %H:%M:%S' returns '2026-03-04 14:30:00'. Must be a non-empty string."
      }
    },
    "required": []
  }
})




In [71]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

# Now lets call the chat module

In [40]:
messages = []
add_user_message(messages, "What is the current time? format it as HH:MM:SS")

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_time_schema]
    )

add_assistant_message(messages, response.content)

print(messages)

[{'role': 'user', 'content': 'What is the current time? format it as HH:MM:SS'}, {'role': 'assistant', 'content': [TextBlock(citations=None, text="I'll get the current time formatted as HH:MM:SS for you.", type='text'), ToolUseBlock(id='toolu_01QVKyBTYpDYCbHCPEz2kDBE', input={'dt_format': '%H:%M:%S'}, name='get_current_time', type='tool_use', caller={'type': 'direct'})]}]


# Now lets retrieve the Input args given by claude

In [41]:
response.content[1].input

{'dt_format': '%H:%M:%S'}

In [43]:
res = get_current_time(**response.content[1].input)

# Now lets give the model the results.

In [44]:
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": response.content[1].id,
        "content": res,
        "is_error": False
    }]
})

In [45]:
response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_time_schema]
    )



In [48]:
response.content[0].text

'The current time is **18:24:32**.'

# Now Lets Bring all this together 

## Let's improve helper functions

In [67]:
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
    "role": "user",
    "content": message.content if isinstance(message, Message) else message
    }
    messages.append(user_message)

def add_assistant_message(messages, message):
    assistant_message = {
    "role": "assistant",
    "content": message.content if isinstance(message, Message) else message
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
        "tools": tools
    }
    
    if system:
        params["system"] = system
    
    if tools:
        params["tools"] = tools
    
    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]

    )


def run_tool(tool_name, tool_input):
    if tool_name == "get_current_time":
        return get_current_time(**tool_input)
    else:
        raise ValueError(f"Tool {tool_name} not found")
        


def run_tools(message):
    tool_requests = [
        block for block in message.content if block.type == "tool_use"
    ]
    tool_result_blocks = []
    
    for tool_request in tool_requests:

        try:
            tool_output = run_tool(tool_request.name,tool_request.input)
            tool_result_blocks.append({
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False
            })
        except Exception as e:
            tool_result_blocks.append({
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": str(e),
        })
            
    return tool_result_blocks

# Now Lets Create a Conversation Cycle 

In [75]:
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[
        get_current_time_schema,
        add_duration_to_datetime_schema,
        set_reminder_schema,
        batch_tool_schema
                ])
        add_assistant_message(messages, response)
        print(text_from_message(response))
        
        if response.stop_reason != "tool_use":
            break
            
        tool_results = run_tools(response)
        add_user_message(messages, tool_results)
    
    return messages

# Now let's ask claude something in 2 tool calls

In [65]:
messages = []
add_user_message(messages, "What is the current date and time? format it as HH:MM:SS. And then also format it as YYYY-MM-DD")
run_conversation(messages)



I'll get the current date and time in both formats you requested.
The current time formatted as HH:MM:SS is: **21:44:55**

The current date formatted as YYYY-MM-DD is: **2026-03-04**


[{'role': 'user',
  'content': 'What is the current date and time? format it as HH:MM:SS. And then also format it as YYYY-MM-DD'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll get the current date and time in both formats you requested.", type='text'),
   ToolUseBlock(id='toolu_015QzSKft7rosCHFM1HqgxLP', input={'dt_format': '%H:%M:%S'}, name='get_current_time', type='tool_use', caller={'type': 'direct'}),
   ToolUseBlock(id='toolu_01RW74JKm2435554wrxop7te', input={'dt_format': '%Y-%m-%d'}, name='get_current_time', type='tool_use', caller={'type': 'direct'})]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_015QzSKft7rosCHFM1HqgxLP',
    'content': '"21:44:55"',
    'is_error': False},
   {'type': 'tool_result',
    'tool_use_id': 'toolu_01RW74JKm2435554wrxop7te',
    'content': '"2026-03-04"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='The current time formatted as HH:MM:

# Now Let's add Batch tooling so that it can pull multiple tools at the same time

In [78]:
def run_batch(invocations=[]):
    batch_output = []
    
    for invocation in invocations:
        name = invocation["name"]
        args = json.loads(invocation["arguments"])
        
        tool_output = run_tool(name, args)
        
        batch_output.append({
            "tool_name": name,
            "output": tool_output
        })
    
    return batch_output



def run_tool(tool_name, tool_input):
    if tool_name == "get_current_time":
        return get_current_time(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)
    elif tool_name == "batch_tool":
        return run_batch(**tool_input)
    else:
        raise ValueError(f"Tool {tool_name} not found")


In [79]:
messages = []
add_user_message(messages, "What is the current date and time? format it as HH:MM:SS. And then also format it as YYYY-MM-DD. And Set Reminder for my dentist appointment 5 hrs later")
run_conversation(messages)

I'll help you get the current time in both formats and set a reminder for your dentist appointment 5 hours later.
Now let me calculate the time 5 hours later and set your reminder:

----
Setting the following reminder for 2026-03-05T20:14:46:
Dentist appointment
----
Here's the information you requested:

**Current Date and Time:**
- Time (HH:MM:SS format): **15:14:46**
- Date (YYYY-MM-DD format): **2026-03-05**

**Reminder Set:**
I've successfully set a reminder for your dentist appointment 5 hours from now, which will be at **8:14:46 PM on Thursday, March 5, 2026**. You'll receive a notification at that time.


[{'role': 'user',
  'content': 'What is the current date and time? format it as HH:MM:SS. And then also format it as YYYY-MM-DD. And Set Reminder for my dentist appointment 5 hrs later'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll help you get the current time in both formats and set a reminder for your dentist appointment 5 hours later.", type='text'),
   ToolUseBlock(id='toolu_01TmJ49FrxiETequPRPi5v2W', input={'invocations': [{'name': 'get_current_time', 'arguments': '{"dt_format": "%H:%M:%S"}'}, {'name': 'get_current_time', 'arguments': '{"dt_format": "%Y-%m-%d"}'}, {'name': 'get_current_time', 'arguments': '{"dt_format": "%Y-%m-%dT%H:%M:%S"}'}]}, name='batch_tool', type='tool_use', caller={'type': 'direct'})]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01TmJ49FrxiETequPRPi5v2W',
    'content': '[{"tool_name": "get_current_time", "output": "15:14:46"}, {"tool_name": "get_current_time", "output": "2026-03-05"}, {"